In [1]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd


from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report
import optuna


os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']
path = competition_download('ing-hubs-turkiye-datathon')

/home/osman/Desktop/Projects/ING_Datathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [3]:
def recall_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek recall değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki recall oranı.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    P = y_true.sum()

    return float(tp_at_k / P) if P > 0 else 0.0


def lift_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek lift (precision/prevalence) değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki lift değeri.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    precision_at_k = tp_at_k / m
    prevalence = y_true.mean()

    return float(precision_at_k / prevalence) if prevalence > 0 else 0.0


def convert_auc_to_gini(auc):
    """
    ROC AUC skorunu Gini katsayısına dönüştürür.

    Gini katsayısı, ROC AUC skorunun doğrusal bir dönüşümüdür.

    Parametreler:
        auc (float): ROC AUC skoru (0 ile 1 arasında).

    Döndürür:
        float: Gini katsayısı (-1 ile 1 arasında).
    """
    return 2 * auc - 1


def ing_hubs_datathon_metric(y_true, y_prob):
    """
    Gini, recall@10% ve lift@10% metriklerini birleştiren özel bir metrik hesaplar.

    Metrik, her bir skoru bir baseline modelin metrik değerlerine göre oranlar ve aşağıdaki ağırlıkları uygular:
    - Gini: %40
    - Recall@10%: %30
    - Lift@10%: %30

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.

    Döndürür:
        float: Ağırlıklandırılmış bileşik skor.
    """
    # final metrik için ağırlıklar
    score_weights = {
        "gini": 0.4,
        "recall_at_10perc": 0.3,
        "lift_at_10perc": 0.3,
    }

    # baseline modelin her bir metrik için değerleri
    baseline_scores = {
        "roc_auc": 0.6925726757936908,
        "recall_at_10perc": 0.18469015795868773,
        "lift_at_10perc": 1.847159286784029,
    }

    # y_prob tahminleri için metriklerin hesaplanması
    roc_auc = roc_auc_score(y_true, y_prob)
    recall_at_10perc = recall_at_k(y_true, y_prob, k=0.1)
    lift_at_10perc = lift_at_k(y_true, y_prob, k=0.1)

    new_scores = {
        "roc_auc": roc_auc,
        "recall_at_10perc": recall_at_10perc,
        "lift_at_10perc": lift_at_10perc,
    }

    # roc auc değerlerinin gini değerine dönüştürülmesi
    baseline_scores["gini"] = convert_auc_to_gini(baseline_scores["roc_auc"])
    new_scores["gini"] = convert_auc_to_gini(new_scores["roc_auc"])

    # baseline modeline oranlama
    final_gini_score = new_scores["gini"] / baseline_scores["gini"]
    final_recall_score = new_scores["recall_at_10perc"] / baseline_scores["recall_at_10perc"]
    final_lift_score = new_scores["lift_at_10perc"] / baseline_scores["lift_at_10perc"]

    # ağırlıklandırılmış metriğin hesaplanması
    final_score = (
        final_gini_score * score_weights["gini"] +
        final_recall_score * score_weights["recall_at_10perc"] + 
        final_lift_score * score_weights["lift_at_10perc"]
    )
    return final_score


In [4]:
train_data = customers.merge(referance_data, "right", on="cust_id")
test_data = customers.merge(referance_data_test, "right", on="cust_id")

In [5]:
train_data = train_data.drop(["cust_id", "ref_date"], axis=1)
test_data = test_data.drop(["cust_id", "ref_date"], axis=1)

In [9]:
train_data

,gender,age,province,religion,work_type,work_sector,tenure,churn
0,F,64,NOH,U,Part-time,Technology,135,0
1,F,22,ZUI,C,Student,NaN,47,0
2,M,27,ZUI,U,Full-time,Finance,108,1
3,F,40,NOH,U,Unemployed,NaN,187,1
4,F,64,GEL,U,Part-time,Public Sector,218,0
...,...,...,...,...,...,...,...,...
133282,F,54,GEL,C,Part-time,Public Sector,217,0
133283,M,47,GEL,C,Full-time,Public Sector,37,0
133284,F,66,NOB,C,Retired,NaN,227,0
133285,F,31,ZUI,U,Self-employed,Education,156,1


In [10]:
num_cols = [col for col in test_data.columns if test_data[col].dtype != object]
cat_cols = [col for col in test_data.columns if test_data[col].dtype == object]

In [11]:
train_cat_df = pd.get_dummies(train_data[cat_cols], drop_first=True)
test_cat_df = pd.get_dummies(test_data[cat_cols], drop_first=True)

In [12]:
scaler = MinMaxScaler(feature_range=(0,1))
train_num_df = pd.DataFrame(scaler.fit_transform(train_data[num_cols]), columns=num_cols)
test_num_df = pd.DataFrame(scaler.transform(test_data[num_cols]), columns=num_cols)

In [13]:
new_train = pd.concat([train_cat_df, train_num_df,train_data["churn"]], axis=1)
new_test  =pd.concat([test_cat_df, test_num_df], axis=1)

In [14]:
X = new_train.drop("churn", axis=1)
y = new_train["churn"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [16]:
def objective(trial, X, y):
    """
    Optuna'nın her bir denemede çalıştıracağı ve özel metriği maksimize edeceği fonksiyon.
    """
    
    # Hiperparametre Arama Uzayını Tanımla
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'booster': 'gbtree',
        'device': 'gpu',
        'early_stopping_rounds': 50,
        'n_estimators': 1000,
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }

    # Dengesiz veri için kritik olan sınıf ağırlığını hesapla
    scale_pos_weight = np.sum(y == 0) / np.sum(y == 1)
    param['scale_pos_weight'] = scale_pos_weight

    # StratifiedKFold ile Çapraz Doğrulama
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**param, random_state=42)
        
        # Modeli eğit (Early stopping ile aşırı öğrenmeyi engelle)
        model.fit(X_train_fold, y_train_fold,
                  eval_set=[(X_val_fold, y_val_fold)],
                  verbose=False)
        
        preds_proba = model.predict_proba(X_val_fold)[:, 1]
        
        # Özel değerlendirme metriğini kullanarak skoru hesapla
        custom_score = ing_hubs_datathon_metric(y_val_fold, preds_proba)
        scores.append(custom_score)

    # Ortalamayı döndür. Optuna bu değeri maksimize etmeye çalışacak.
    return np.mean(scores)


In [17]:

# =============================================================================
# 5. Optimizasyon Sürecini Başlatma
# =============================================================================
print("--- Optuna Optimizasyonu Başlatılıyor ---")
# 'direction="maximize"' ile özel metriğimizin en yüksek değerini arıyoruz
study = optuna.create_study(direction='maximize')

# Optimizasyonu n_trials kadar deneme ile çalıştır
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, show_progress_bar=True)

print("Optimizasyon tamamlandı.\n")
print("--- En İyi Optimizasyon Sonuçları ---")
best_trial = study.best_trial
print(f"En İyi Değer (Ortalama Özel Metrik): {best_trial.value:.4f}")
print("En İyi Parametreler:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")
print("-" * 30, "\n")



[I 2025-10-10 22:01:17,434] A new study created in memory with name: no-name-7d3d720b-3223-4319-8de3-d263492634f9


--- Optuna Optimizasyonu Başlatılıyor ---


  0%|          | 0/50 [00:00<?, ?it/s]/home/osman/Desktop/Projects/ING_Datathon/.venv/lib/python3.12/site-packages/xgboost/core.py:729: UserWarning: [22:01:25] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Best trial: 0. Best value: 0.342593:   2%|▏         | 1/50 [00:37<30:22, 37.20s/it]

[I 2025-10-10 22:01:54,640] Trial 0 finished with value: 0.34259316537310724 and parameters: {'lambda': 0.0005972877355051952, 'alpha': 0.0022749281544483693, 'max_depth': 8, 'eta': 0.04938643979589921, 'gamma': 0.3274455877690994, 'colsample_bytree': 0.864592984532513, 'subsample': 0.8890635943078403, 'min_child_weight': 3}. Best is trial 0 with value: 0.34259316537310724.


Best trial: 0. Best value: 0.342593:   4%|▍         | 2/50 [01:11<28:27, 35.57s/it]

...

[I 2025-10-10 22:12:28,308] Trial 47 finished with value: 0.40399832571451755 and parameters: {'lambda': 0.160546248344729, 'alpha': 0.0004573410882974458, 'max_depth': 4, 'eta': 0.020540945224319457, 'gamma': 7.796029835841143e-07, 'colsample_bytree': 0.7558152524867293, 'subsample': 0.5325514521058821, 'min_child_weight': 7}. Best is trial 46 with value: 0.42293052787389385.


Best trial: 46. Best value: 0.422931:  98%|█████████▊| 49/50 [11:43<00:15, 15.26s/it]

[I 2025-10-10 22:13:01,170] Trial 48 finished with value: 0.341220547929659 and parameters: {'lambda': 0.008107085455016259, 'alpha': 6.888036001466302e-05, 'max_depth': 8, 'eta': 0.031342067313308755, 'gamma': 2.0108852600855539e-07, 'colsample_bytree': 0.7816126028921524, 'subsample': 0.6603118886107928, 'min_child_weight': 7}. Best is trial 46 with value: 0.42293052787389385.


Best trial: 46. Best value: 0.422931: 100%|██████████| 50/50 [11:46<00:00, 14.13s/it]

[I 2025-10-10 22:13:04,187] Trial 49 finished with value: 0.4204418177398973 and parameters: {'lambda': 0.023489132414419744, 'alpha': 0.0027062721118520953, 'max_depth': 3, 'eta': 0.0160187627994233, 'gamma': 9.637425207813684e-06, 'colsample_bytree': 0.7256020519422514, 'subsample': 0.6087212912013155, 'min_child_weight': 6}. Best is trial 46 with value: 0.42293052787389385.
Optimizasyon tamamlandı.

--- En İyi Optimizasyon Sonuçları ---
En İyi Değer (Ortalama Özel Metrik): 0.4229
En İyi Parametreler:
  lambda: 0.0074970449541763235
  alpha: 0.002079300057047954
  max_depth: 3
  eta: 0.020467429767561853
  gamma: 8.099726857823711e-07
  colsample_bytree: 0.7372957130727487
  subsample: 0.6114801781638591
  min_child_weight: 7
------------------------------ 



In [18]:

# =============================================================================
# 6. Final Modelin Eğitilmesi ve Değerlendirilmesi
# =============================================================================
print("--- Final Model Eğitiliyor ve Değerlendiriliyor ---")
# Optuna'nın bulduğu en iyi parametreleri al
best_params = best_trial.params

# Optimizasyon dışında kalan sabit parametreleri ekle
best_params['scale_pos_weight'] = np.sum(y_train == 0) / np.sum(y_train == 1)
best_params['n_estimators'] = 2000  # Early stopping için yüksek bir değer
best_params['random_state'] = 42
best_params['objective'] = 'binary:logistic'
best_params["early_stopping_rounds"] = 50

# Final modeli en iyi parametrelerle oluştur
final_model = xgb.XGBClassifier(**best_params)

# Final modelin eğitiminde de early stopping kullanmak iyi bir pratiktir.
# Bunun için eğitim verisinden küçük bir validasyon seti ayırabiliriz.
X_train_part, X_val_part, y_train_part, y_val_part = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

final_model.fit(X_train_part, y_train_part,
                eval_set=[(X_val_part, y_val_part)],
                verbose=False)

# Daha önce hiç görülmemiş TEST VERİSİ üzerinde tahmin yap
y_pred_proba_test = final_model.predict_proba(X_test)[:, 1]

# Test seti üzerinde özel metrik skorunu hesapla
final_custom_score = ing_hubs_datathon_metric(y_test, y_pred_proba_test)
print(f"Test Seti Üzerindeki Özel Metrik Skoru: {final_custom_score:.4f}\n")

# Özel metriği oluşturan alt metriklerin dökümünü de alalım
test_auc = roc_auc_score(y_test, y_pred_proba_test)
test_gini = convert_auc_to_gini(test_auc)
test_recall10 = recall_at_k(y_test, y_pred_proba_test, k=0.1)
test_lift10 = lift_at_k(y_test, y_pred_proba_test, k=0.1)

print("--- Test Seti Detaylı Metrikler ---")
print(f"Gini: {test_gini:.4f}")
print(f"Recall@10%: {test_recall10:.4f}")
print(f"Lift@10%: {test_lift10:.4f}\n")

# Sınıflandırma raporu için bir eşik değeri belirleyelim (örn: 0.5)
y_pred_class_test = (y_pred_proba_test > 0.5).astype(int)
print("--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---")
print(classification_report(y_test, y_pred_class_test))
print("=" * 70)

--- Final Model Eğitiliyor ve Değerlendiriliyor ---
Test Seti Üzerindeki Özel Metrik Skoru: 0.4080

--- Test Seti Detaylı Metrikler ---
Gini: 0.0461
Recall@10%: 0.1109
Lift@10%: 1.1086

--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---
              precision    recall  f1-score   support

           0       0.86      0.56      0.68     28604
           1       0.15      0.47      0.23      4718

    accuracy                           0.54     33322
   macro avg       0.51      0.51      0.45     33322
weighted avg       0.76      0.54      0.61     33322



In [ ]:
best_params.pop("early_stopping_rounds")

In [ ]:
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X,y)

In [ ]:
sample_submission["churn"] = final_model.predict(new_test)

In [ ]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='xgb with Optuna kfold', 
    competition='ing-hubs-turkiye-datathon'
)